<div style="background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);padding:40px;border-radius:12px;color:white;text-align:center;">
<h1 style="color:#e94560;font-size:2.2em;">🦾 Agent IA — Boite Mail Thunderbird</h1>
<h2 style="color:#a8dadc;">TP Operationnel — Gmail IMAP + Ollama local</h2>
<p style="color:#ccc;">Lecture reelle des mails · Analyse PJ · Brouillons de reponse</p>
<hr style="border-color:#e94560;margin:20px 0;">
<div style="display:flex;justify-content:space-around;flex-wrap:wrap;gap:10px;font-size:.9em;">
<span style="background:rgba(233,69,96,.2);border:1px solid #e94560;padding:6px 14px;border-radius:20px;">📬 Etape 1 — Config Gmail</span>
<span style="background:rgba(168,218,220,.2);border:1px solid #a8dadc;padding:6px 14px;border-radius:20px;">🔌 Etape 2 — Connexion IMAP</span>
<span style="background:rgba(255,200,100,.2);border:1px solid #ffc864;padding:6px 14px;border-radius:20px;">🧠 Etape 3 — Triage Ollama</span>
<span style="background:rgba(46,204,113,.2);border:1px solid #2ecc71;padding:6px 14px;border-radius:20px;">✉️ Etape 4 — Brouillons</span>
</div></div>

---
## Etape 1 — Prerequis Gmail : App Password

Gmail bloque les connexions IMAP avec le mot de passe principal.
Il faut creer un **App Password** dedie :

### Procedure (2 minutes)
1. [myaccount.google.com/security](https://myaccount.google.com/security)
2. **Activer la validation en 2 etapes** si ce n'est pas fait
3. Chercher **"App passwords"**
4. Nom de l'app : `ThunderAgent` puis **Create**
5. Copier le mot de passe a 16 caracteres (ex: `abcdefghijklmnop`)

### Activer IMAP dans Gmail
Gmail → Settings → **See all settings** → onglet **Forwarding and POP/IMAP** → **Enable IMAP**

### Variables d'environnement (Terminal avant Jupyter)
```bash
export GMAIL_USER="votre.adresse@gmail.com"
export GMAIL_APP_PASSWORD="abcdefghijklmnop"
```

> **Confidentialite** : tous les traitements sont 100% locaux.
> Ollama tourne sur votre machine, aucun mail n'est envoye vers un service cloud.

In [15]:
# %pip install PyPDF2 openpyxl python-docx plotly pandas

import os, imaplib, email, json, io, urllib.request
from email.header import decode_header
from typing       import List, Dict, Tuple
from datetime     import datetime
import pandas     as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLORS = {'primary':'#e94560','secondary':'#0f3460',
          'accent':'#ffc864','success':'#2ecc71','neutral':'#95a5a6'}
URGENCY_EMOJI = {"critique":"🔴","haute":"🟠","normale":"🟡","faible":"⚪"}

print("Imports OK")

Imports OK


---
## Etape 2 — Configuration

In [ ]:
GMAIL_USER         = os.getenv("GMAIL_USER",         "anne.centrale.mediterranee@gmail.com")
GMAIL_APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD", "karx ixtp gzna vjog")

OLLAMA_URL   = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:3b"    # llama3.1:8b / mistral-nemo:12b / qwen2.5:7b

NB_MAILS_MAX = 10
UNREAD_ONLY  = False

print("Configuration :")
print(f"  Gmail    : {GMAIL_USER}")
print(f"  Password : {'OK' if GMAIL_APP_PASSWORD else 'MANQUANT — voir Etape 1'}")
print(f"  Ollama   : {OLLAMA_URL}")
print(f"  Modele   : {OLLAMA_MODEL}")

Configuration :
  Gmail    : anne.centrale.mediterranee@gmail.com
  Password : OK
  Ollama   : http://localhost:11434
  Modele   : llama3.2:3b


---
## Etape 3 — Connexion IMAP Gmail

In [17]:
class GmailReader:

    IMAP_HOST = "imap.gmail.com"
    IMAP_PORT = 993

    def __init__(self, user, password):
        self.user, self.password, self.conn = user, password, None

    def connect(self):
        try:
            self.conn = imaplib.IMAP4_SSL(self.IMAP_HOST, self.IMAP_PORT)
            self.conn.login(self.user, self.password)
            print(f"Connecte a Gmail ({self.user})")
            return True
        except imaplib.IMAP4.error as e:
            print(f"Erreur IMAP : {e}")
            print("  -> App Password correct ? IMAP active dans Gmail ?")
            return False
        except Exception as e:
            print(f"Erreur reseau : {e}")
            return False

    def disconnect(self):
        if self.conn:
            try: self.conn.logout()
            except: pass

    @staticmethod
    def _decode(value):
        if not value: return ""
        parts = decode_header(value)
        result = []
        for part, enc in parts:
            if isinstance(part, bytes):
                result.append(part.decode(enc or "utf-8", errors="replace"))
            else:
                result.append(str(part))
        return " ".join(result).strip()

    def _get_body(self, msg):
        body = ""
        if msg.is_multipart():
            for part in msg.walk():
                ct = part.get_content_type()
                cd = str(part.get("Content-Disposition",""))
                if ct == "text/plain" and "attachment" not in cd:
                    charset = part.get_content_charset() or "utf-8"
                    try:
                        body += part.get_payload(decode=True).decode(charset, errors="replace")
                    except: pass
        else:
            charset = msg.get_content_charset() or "utf-8"
            try: body = msg.get_payload(decode=True).decode(charset, errors="replace")
            except: body = ""
        return body.replace("\r\n","\n").replace("\r","\n")[:4000]

    def _get_attachments(self, msg):
        atts = []
        for part in msg.walk():
            fn = self._decode(part.get_filename())
            if not fn: continue
            payload = part.get_payload(decode=True) or b""
            atts.append({"filename": fn,
                          "content_type": part.get_content_type(),
                          "data": payload,
                          "size_kb": round(len(payload)/1024, 1)})
        return atts

    def fetch(self, n=10, mailbox="INBOX", unread_only=False):
        if not self.conn: print("Non connecte"); return []
        self.conn.select(mailbox)
        criteria = "UNSEEN" if unread_only else "ALL"
        _, msg_ids = self.conn.search(None, criteria)
        ids = msg_ids[0].split()
        if not ids: print(f"Aucun mail ({criteria})"); return []
        ids_recent = ids[-n:][::-1]
        print(f"Boite : {len(ids)} mails total, lecture de {len(ids_recent)}")
        mails = []
        for i, uid in enumerate(ids_recent):
            try:
                _, data = self.conn.fetch(uid, "(RFC822)")
                msg = email.message_from_bytes(data[0][1])
                mails.append({
                    "uid":         uid.decode(),
                    "from":        self._decode(msg["From"]),
                    "to":          self._decode(msg["To"]),
                    "subject":     self._decode(msg["Subject"]) or "(Sans objet)",
                    "date":        msg["Date"] or "",
                    "body":        self._get_body(msg),
                    "attachments": self._get_attachments(msg),
                })
                print(f"  [{i+1}/{len(ids_recent)}] {mails[-1]['subject'][:58]}")
            except Exception as e:
                print(f"  UID {uid} erreur : {e}")
        return mails

    def list_folders(self):
        _, folders = self.conn.list()
        return [f.decode().split('"."')[-1].strip().strip('"') for f in folders if f]

print("GmailReader defini")

GmailReader defini


### Test de connexion

In [22]:
GMAIL_USER         = "anne.centrale.mediterranee@gmail.com"
GMAIL_APP_PASSWORD = "karx ixtp gzna vjog"

In [23]:
reader = GmailReader(GMAIL_USER, GMAIL_APP_PASSWORD)
if reader.connect():
    print()
    print("Dossiers disponibles :")
    try:
        for folder in reader.list_folders()[:12]:
            print(f"  {folder}")
    except: pass
else:
    print()
    print("Solutions :")
    print("  1. App Password correct ? (16 car, sans espaces)")
    print("  2. IMAP active dans Gmail Settings ?")
    print("  3. Compte Workspace ? -> contacter l'admin")

Connecte a Gmail (anne.centrale.mediterranee@gmail.com)

Dossiers disponibles :
  (\HasNoChildren) "/" "INBOX
  (\HasChildren \Noselect) "/" "[Gmail]
  (\Drafts \HasNoChildren) "/" "[Gmail]/Brouillons
  (\HasNoChildren \Trash) "/" "[Gmail]/Corbeille
  (\HasNoChildren \Important) "/" "[Gmail]/Important
  (\HasNoChildren \Sent) "/" "[Gmail]/Messages envoy&AOk-s
  (\HasNoChildren \Junk) "/" "[Gmail]/Spam
  (\Flagged \HasNoChildren) "/" "[Gmail]/Suivis
  (\All \HasNoChildren) "/" "[Gmail]/Tous les messages


In [24]:
print(f"User     : '{GMAIL_USER}'")
print(f"Password : '{GMAIL_APP_PASSWORD}'")
print(f"Longueur : {len(GMAIL_APP_PASSWORD)} caractères")

User     : 'anne.centrale.mediterranee@gmail.com'
Password : 'karx ixtp gzna vjog'
Longueur : 19 caractères


In [25]:
MAILS = []
if reader.conn:
    MAILS = reader.fetch(n=NB_MAILS_MAX, unread_only=UNREAD_ONLY)
    reader.disconnect()
    print(f"\n{len(MAILS)} mails charges")
else:
    print("Connexion non etablie")

if MAILS:
    print()
    print(f"{'#':<3} {'Expediteur':<30} {'Objet':<45} {'PJ':>3}")
    print("-"*83)
    for i, m in enumerate(MAILS):
        print(f"{i+1:<3} {m['from'][:28]:<30} {m['subject'][:43]:<45} {len(m['attachments']):>3}")

Boite : 3 mails total, lecture de 3
  [1/3] Alerte de sécurité
  [2/3] Validation en deux étapes activée
  [3/3] Informations sur votre nouveau compte Google

3 mails charges

#   Expediteur                     Objet                                          PJ
-----------------------------------------------------------------------------------
1   Google <no-reply@accounts.go   Alerte de sécurité                              0
2   Google <no-reply@accounts.go   Validation en deux étapes activée               0
3   Google <no-reply@google.com>   Informations sur votre nouveau compte Googl     0


---
## Etape 4 — Parseur de pieces jointes

In [26]:
class AttachmentParser:

    MAX_CHARS = 3000

    def parse(self, att):
        ct = att["content_type"].lower()
        fn = att["filename"].lower()
        try:
            if "pdf"   in ct or fn.endswith(".pdf"):             return self._pdf(att)
            if "csv"   in ct or fn.endswith(".csv"):             return self._csv(att)
            if "excel" in ct or fn.endswith((".xlsx",".xls")):  return self._excel(att)
            if "word"  in ct or fn.endswith((".docx",".doc")): return self._docx(att)
            if "text"  in ct or fn.endswith(".txt"):             return self._text(att)
            if ct.startswith("image/"):
                return f"[IMAGE: {att['filename']} ({att['size_kb']}KB)]"
            return f"[{att['filename']} type {ct} non supporte]"
        except Exception as e:
            return f"[Erreur {att['filename']}: {e}]"

    def _pdf(self, att):
        try:
            import PyPDF2
            reader = PyPDF2.PdfReader(io.BytesIO(att["data"]))
            text   = "\n".join(p.extract_text() or "" for p in reader.pages)
            trunc  = "...[tronque]" if len(text) > self.MAX_CHARS else ""
            return f"[PDF: {att['filename']} ({len(reader.pages)} pages)]\n{text[:self.MAX_CHARS]}{trunc}"
        except ImportError:
            return f"[PDF {att['filename']} -> pip install PyPDF2]"

    def _csv(self, att):
        text = att["data"].decode("utf-8", errors="replace")
        try:
            df   = pd.read_csv(io.StringIO(text))
            info = (f"[CSV: {att['filename']} - {len(df)}L x {len(df.columns)}C]\n"
                    f"Colonnes: {list(df.columns)}\n"
                    f"Apercu:\n{df.head(5).to_string()}\n"
                    f"Stats:\n{df.describe().round(3).to_string()}")
            return info[:self.MAX_CHARS]
        except:
            return f"[CSV {att['filename']}]\n{text[:self.MAX_CHARS]}"

    def _excel(self, att):
        try:
            df   = pd.read_excel(io.BytesIO(att["data"]))
            info = (f"[Excel: {att['filename']} - {df.shape[0]}L x {df.shape[1]}C]\n"
                    f"Colonnes: {list(df.columns)}\n"
                    f"Apercu:\n{df.head(8).to_string()}")
            return info[:self.MAX_CHARS]
        except Exception as e:
            return f"[Excel {att['filename']} -> pip install openpyxl : {e}]"

    def _docx(self, att):
        try:
            from docx import Document
            doc  = Document(io.BytesIO(att["data"]))
            text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
            trunc = "...[tronque]" if len(text) > self.MAX_CHARS else ""
            return f"[Word: {att['filename']}]\n{text[:self.MAX_CHARS]}{trunc}"
        except ImportError:
            return f"[Word {att['filename']} -> pip install python-docx]"

    def _text(self, att):
        text = att["data"].decode("utf-8", errors="replace")
        return f"[Texte: {att['filename']}]\n{text[:self.MAX_CHARS]}"

PARSER   = AttachmentParser()
pj_total = sum(len(m["attachments"]) for m in MAILS)
print(f"AttachmentParser pret - {pj_total} PJ dans {len(MAILS)} mails")
for m in MAILS:
    for att in m["attachments"]:
        print(f"  {att['filename']} ({att['size_kb']} KB) - {att['content_type']}")

AttachmentParser pret - 0 PJ dans 3 mails


---
## Etape 5 — Ollama

In [27]:
def check_ollama():
    try:
        req  = urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=3)
        data = json.loads(req.read())
        return True, [m["name"] for m in data.get("models", [])]
    except:
        return False, []

def ollama_chat(messages, tools=None, temperature=0.2):
    payload = {"model": OLLAMA_MODEL, "messages": messages,
               "stream": False, "options": {"temperature": temperature}}
    if tools:
        payload["tools"] = tools
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/chat",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read())

ollama_ok, installed = check_ollama()
if ollama_ok:
    print(f"Ollama actif - {len(installed)} modele(s) :")
    for m in installed:
        tag = " <- selectionne" if OLLAMA_MODEL in m else ""
        print(f"  {m}{tag}")
    if not any(OLLAMA_MODEL in m for m in installed):
        print(f"\nModele '{OLLAMA_MODEL}' absent -> ollama pull {OLLAMA_MODEL}")
    else:
        resp = ollama_chat([{"role":"user","content":"Reponds uniquement : OK pret"}])
        print(f"\nTest : {resp['message']['content'].strip()}")
else:
    print("Ollama non disponible")
    print("  Terminal : ollama serve")
    print(f"  Puis     : ollama pull {OLLAMA_MODEL}")

Ollama actif - 6 modele(s) :
  llama3.2:3b <- selectionne
  gemma4:31b-cloud
  qwen3.5:cloud
  qwen3.5:latest
  gemma3:4b
  deepseek-r1:latest

Test : OK, je suis prêt ! Qu'est-ce que je peux faire pour vous ?


---
## Etape 6 — Triage par urgence (LLM)

In [28]:
TRIAGE_SYSTEM = (
    "Tu es un assistant expert en gestion de boite mail professionnelle. "
    "Tu analyses les emails et evalues leur urgence. "
    "Tu reponds UNIQUEMENT en JSON valide, sans aucun texte autour. "
    "Langue : francais."
)

def trier_mail(mail):
    pj_names = [a["filename"] for a in mail["attachments"]]
    pj_info  = f"\nPieces jointes : {', '.join(pj_names)}" if pj_names else ""

    schema = (
        "{\n"
        '  "urgence": "critique" | "haute" | "normale" | "faible",\n'
        '  "categorie": "action_requise" | "information" | "question" | "spam" | "commercial",\n'
        '  "resume": "resume en 1 phrase (max 120 car)",\n'
        '  "action_suggeree": "action a faire (max 100 car)",\n'
        '  "delai_reponse": "immediat" | "aujourd_hui" | "cette_semaine" | "aucun"\n'
        "}"
    )

    prompt = (
        f"Analyse cet email et retourne exactement ce JSON :\n{schema}\n\n"
        f"De      : {mail['from']}\n"
        f"Objet   : {mail['subject']}\n"
        f"Date    : {mail['date']}{pj_info}\n\n"
        f"Corps :\n{mail['body'][:1500]}\n\n"
        "JSON uniquement :"
    )

    resp = ollama_chat(
        messages=[
            {"role": "system", "content": TRIAGE_SYSTEM},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.05,
    )
    raw = resp["message"]["content"].strip()
    raw = raw.replace("```json","").replace("```","").strip()
    s   = raw.find("{")
    e   = raw.rfind("}") + 1
    if s >= 0 and e > s:
        raw = raw[s:e]
    try:
        return json.loads(raw)
    except:
        return {
            "urgence":        "normale",
            "categorie":      "information",
            "resume":         mail["subject"][:120],
            "action_suggeree":"Lire le mail",
            "delai_reponse":  "cette_semaine",
        }

print("Triage defini - temperature=0.05 pour JSON stable")


Triage defini - temperature=0.05 pour JSON stable


In [29]:
if not MAILS:
    print("Aucun mail charge - relancer Etape 3")
elif not ollama_ok:
    print("Ollama non disponible - relancer Etape 5")
else:
    print(f"Triage de {len(MAILS)} mails avec {OLLAMA_MODEL}...\n")
    for i, mail in enumerate(MAILS):
        subj = mail['subject'][:52]
        print(f"  [{i+1}/{len(MAILS)}] {subj}...", end=" ", flush=True)
        try:
            mail["analyse"] = trier_mail(mail)
            print(URGENCY_EMOJI.get(mail["analyse"].get("urgence","normale"),"🟡"))
        except Exception as e:
            print(f"erreur : {e}")
            mail["analyse"] = {"urgence":"normale","categorie":"information",
                                "resume":mail["subject"][:120],
                                "action_suggeree":"Lire le mail",
                                "delai_reponse":"cette_semaine"}
    print("\nTriage termine")

Triage de 3 mails avec llama3.2:3b...

  [1/3] Alerte de sécurité... 🟡
  [2/3] Validation en deux étapes activée... 🟡
  [3/3] Informations sur votre nouveau compte Google... 🟡

Triage termine


### Dashboard de triage

In [30]:
if MAILS and any("analyse" in m for m in MAILS):
    rows = [{"from":     m["from"][:35],
             "subject":  m["subject"][:50],
             "urgence":  m.get("analyse",{}).get("urgence","normale"),
             "categorie":m.get("analyse",{}).get("categorie","information"),
             "delai":    m.get("analyse",{}).get("delai_reponse","cette_semaine"),
             "resume":   m.get("analyse",{}).get("resume",""),
             "action":   m.get("analyse",{}).get("action_suggeree",""),
             "nb_pj":    len(m["attachments"])} for m in MAILS]
    df = pd.DataFrame(rows)

    urg_order  = ["critique","haute","normale","faible"]
    urg_colors = {"critique":"#e94560","haute":"#f39c12","normale":"#3498db","faible":"#95a5a6"}
    urg_counts = df["urgence"].value_counts().reindex(urg_order, fill_value=0)
    cat_counts = df["categorie"].value_counts()
    del_order  = ["immediat","aujourd_hui","cette_semaine","aucun"]
    del_colors = {"immediat":"#e94560","aujourd_hui":"#f39c12",
                  "cette_semaine":"#3498db","aucun":"#95a5a6"}
    del_counts = df["delai"].value_counts().reindex(del_order, fill_value=0)

    fig = make_subplots(rows=1, cols=3,
        subplot_titles=("Urgence","Categorie","Delai reponse"),
        specs=[[{"type":"pie"},{"type":"bar"},{"type":"bar"}]])

    fig.add_trace(go.Pie(
        labels=urg_counts.index.tolist(), values=urg_counts.values.tolist(),
        marker=dict(colors=[urg_colors.get(u,"gray") for u in urg_counts.index]),
        hole=0.4, textinfo="label+percent"), row=1, col=1)

    fig.add_trace(go.Bar(x=cat_counts.values, y=cat_counts.index, orientation="h",
        marker_color=COLORS["secondary"]), row=1, col=2)

    fig.add_trace(go.Bar(x=del_order, y=del_counts.values,
        marker_color=[del_colors[d] for d in del_order]), row=1, col=3)

    fig.update_layout(height=380, template="plotly_white", showlegend=False,
        title=f"Dashboard boite mail - {len(MAILS)} mails analyses par {OLLAMA_MODEL}")
    fig.show()

    # Tableau trie
    print("\nTRIAGE PAR URGENCE\n")
    rank = {"critique":0,"haute":1,"normale":2,"faible":3}
    srt  = sorted(MAILS, key=lambda m: rank.get(m.get("analyse",{}).get("urgence","normale"),2))
    print(f"{'#':<3} {'U':<2} {'Expediteur':<28} {'Objet':<38} {'Action':<32} {'Delai':<13}")
    print("-"*118)
    for i, m in enumerate(srt):
        a   = m.get("analyse", {})
        urg = a.get("urgence","normale")
        src = m["from"].split("<")[0].strip()[:26]
        print(f"{i+1:<3} {URGENCY_EMOJI.get(urg,'🟡'):<2} {src:<28} {m['subject'][:36]:<38}"
              f" {a.get('action_suggeree','')[:30]:<32} {a.get('delai_reponse',''):<13}")


TRIAGE PAR URGENCE

#   U  Expediteur                   Objet                                  Action                           Delai        
----------------------------------------------------------------------------------------------------------------------
1   🟡  Google                       Alerte de sécurité                     Lire le mail                     cette_semaine
2   🟡  Google                       Validation en deux étapes activée      Lire le mail                     cette_semaine
3   🟡  Google                       Informations sur votre nouveau compt   Lire le mail                     cette_semaine


---
## Etape 7 — Analyse des pieces jointes

In [31]:
def analyser_pj_mail(mail):
    if not mail["attachments"]:
        return "Aucune piece jointe."
    results = []
    for att in mail["attachments"]:
        print(f"  Parsing {att['filename']} ({att['size_kb']} KB)...")
        results.append(PARSER.parse(att))
    return "\n\n".join(results)

mails_avec_pj = [m for m in MAILS if m["attachments"]]
print(f"{len(mails_avec_pj)} mail(s) avec pieces jointes sur {len(MAILS)}\n")
for m in mails_avec_pj[:3]:
    print(f"{'='*65}")
    print(f"{m['subject']}")
    print(f"De : {m['from']}")
    print("="*65)
    contenu = analyser_pj_mail(m)
    m["pj_content"] = contenu
    print(contenu[:800])
    if len(contenu) > 800:
        print(f"  ...[{len(contenu)-800} caracteres de plus]")
    print()

0 mail(s) avec pieces jointes sur 3



---
## Etape 8 — Generation des brouillons de reponse

In [32]:
REPLY_SYSTEM = (
    "Tu es un assistant professionnel qui redige des reponses d'email en francais. "
    "Style : professionnel, concis, chaleureux. "
    "Tu utilises les informations du mail original ET des pieces jointes pour une reponse precise. "
    "Si des informations manquent, indique-le clairement."
)

def generer_reponse(mail):
    a         = mail.get("analyse", {})
    pj_text   = mail.get("pj_content", "")
    pj_section = f"\n\nCONTENU DES PIECES JOINTES :\n{pj_text[:2000]}" if pj_text else ""

    sender_parts = mail["from"].split("<")[0].strip().split()
    sender_name  = sender_parts[0] if sender_parts else "Madame/Monsieur"

    prompt = (
        "Redige une reponse professionnelle a cet email.\n\n"
        "ANALYSE IA :\n"
        f"- Urgence    : {a.get('urgence','normale')}\n"
        f"- Categorie  : {a.get('categorie','information')}\n"
        f"- Action     : {a.get('action_suggeree','')}\n\n"
        "EMAIL ORIGINAL :\n"
        f"De      : {mail['from']}\n"
        f"Objet   : {mail['subject']}\n"
        f"Date    : {mail['date']}\n\n"
        f"Corps :\n{mail['body'][:2000]}"
        f"{pj_section}\n\n"
        "INSTRUCTIONS :\n"
        f"- Commence par : Bonjour {sender_name},\n"
        "- Reponds precisement au contenu du mail\n"
        "- Si action requise : confirme ce que tu vas faire et le delai\n"
        "- Si question : reponds directement\n"
        "- Conclus avec une formule de politesse professionnelle\n"
        "- Signature : [Votre nom]\n"
        "\nBrouillon :"
    )

    resp = ollama_chat(
        messages=[
            {"role": "system", "content": REPLY_SYSTEM},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.5,
    )
    return resp["message"]["content"].strip()

print("generer_reponse() defini")


generer_reponse() defini


In [33]:
rank = {"critique":0,"haute":1,"normale":2,"faible":3}
prioritaires = sorted(
    [m for m in MAILS
     if m.get("analyse",{}).get("urgence") in ("critique","haute")
     or m.get("analyse",{}).get("categorie") == "action_requise"],
    key=lambda m: rank.get(m.get("analyse",{}).get("urgence","normale"),2)
) or MAILS[:3]

print(f"Generation de brouillons pour {len(prioritaires)} mail(s) prioritaires\n")

BROUILLONS = []
for i, mail in enumerate(prioritaires):
    a   = mail.get("analyse", {})
    urg = a.get("urgence","normale")
    print(f"\n{'='*65}")
    print(f"[{i+1}/{len(prioritaires)}] {mail['subject']}")
    print(f"De      : {mail['from']}")
    print(f"Urgence : {URGENCY_EMOJI.get(urg,'🟡')} {urg.upper()}")
    print(f"Resume  : {a.get('resume','')}")
    if mail["attachments"] and "pj_content" not in mail:
        print("Parsing pieces jointes...")
        mail["pj_content"] = analyser_pj_mail(mail)
    print(f"\nGeneration ({OLLAMA_MODEL})...\n")
    try:
        brouillon = generer_reponse(mail)
        mail["brouillon"] = brouillon
        BROUILLONS.append({"mail": mail, "brouillon": brouillon})
        print("--- BROUILLON " + "-"*50)
        for line in brouillon.split("\n"):
            print(f"  {line}")
        print("-"*65)
    except Exception as e:
        print(f"Erreur : {e}")

print(f"\n{len(BROUILLONS)} brouillon(s) genere(s)")

Generation de brouillons pour 3 mail(s) prioritaires


[1/3] Alerte de sécurité
De      : Google <no-reply@accounts.google.com>
Urgence : 🟡 NORMALE
Resume  : Alerte de sécurité

Generation (llama3.2:3b)...

--- BROUILLON --------------------------------------------------
  Voici la réponse professionnelle :
  
  Bonjour Google,
  
  Je vous remercie pour votre alerte de sécurité. Je vais vérifier et sécuriser mon compte dès maintenant.
  
  Pour ce faire, je vais consulter l'activité liée à la sécurité de mon compte ici : https://myaccount.google.com/notifications. Je vais également vérifier que le mot de passe d'applications créé pour moi est correct.
  
  Je vous remercie encore pour votre vigilance et votre protection des utilisateurs.
  
  Cordialement,
  [Votre nom]
-----------------------------------------------------------------

[2/3] Validation en deux étapes activée
De      : Google <no-reply@accounts.google.com>
Urgence : 🟡 NORMALE
Resume  : Validation en deux étapes activée

---
## Etape 9 — Export des brouillons

In [34]:
def export_html(brouillons, filename="brouillons_agent.html"):
    urg_col = {"critique":"#e94560","haute":"#f39c12",
               "normale":"#3498db","faible":"#95a5a6"}
    cards = []
    for b in brouillons:
        m   = b["mail"]
        a   = m.get("analyse", {})
        urg = a.get("urgence","normale")
        col = urg_col.get(urg,"#95a5a6")
        emo = URGENCY_EMOJI.get(urg,"🟡")
        pj  = ", ".join(att["filename"] for att in m["attachments"]) or "aucune"
        body_html = b["brouillon"].replace("&","&amp;").replace("<","&lt;").replace("\n","<br>")
        cards.append(
            f'<div style="background:white;border-left:5px solid {col};border-radius:8px;'
            f'padding:20px;margin-bottom:24px;box-shadow:0 2px 8px rgba(0,0,0,.08);">'
            f'<div style="display:flex;justify-content:space-between;margin-bottom:12px;">'
            f'<b style="font-size:1.05em;">{emo} {m["subject"]}</b>'
            f'<span style="background:{col};color:white;padding:3px 10px;border-radius:12px;'
            f'font-size:.85em;">{urg.upper()}</span></div>'
            f'<div style="color:#666;font-size:.88em;margin-bottom:10px;">'
            f'<b>De :</b> {m["from"]}<br><b>Date :</b> {m["date"]}<br><b>PJ :</b> {pj}</div>'
            f'<div style="background:#f0f4f8;border-radius:6px;padding:10px;margin-bottom:12px;font-size:.88em;">'
            f'<b>Resume IA :</b> {a.get("resume","")}<br>'
            f'<b>Action :</b> {a.get("action_suggeree","")}</div>'
            f'<div style="background:#fffbea;border:1px solid #ffc864;border-radius:6px;padding:15px;">'
            f'<b style="color:#f39c12;">Brouillon de reponse</b><br><br>'
            f'<div style="white-space:pre-wrap;line-height:1.6;">{body_html}</div>'
            f'</div></div>'
        )
    ts   = datetime.now().strftime("%d/%m/%Y a %H:%M")
    html = (
        '<!DOCTYPE html><html lang="fr"><head><meta charset="UTF-8">'
        '<title>Brouillons Agent Mail</title>'
        '<style>body{font-family:-apple-system,sans-serif;max-width:900px;'
        'margin:40px auto;padding:0 20px;background:#f0f2f5;}</style></head><body>'
        f'<h1 style="color:#1a1a2e;">Brouillons Agent Mail</h1>'
        f'<p style="color:#666;">Genere le {ts} | {len(brouillons)} brouillon(s) | {OLLAMA_MODEL}</p>'
        f'<hr>{"".join(cards)}</body></html>'
    )
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Export HTML : {filename}")

def export_txt(brouillons, filename="brouillons_agent.txt"):
    ts    = datetime.now().strftime("%d/%m/%Y %H:%M")
    lines = [f"BROUILLONS AGENT MAIL - {ts}", "="*65]
    for i, b in enumerate(brouillons):
        m = b["mail"]
        a = m.get("analyse", {})
        lines += [
            f"\n[{i+1}] {URGENCY_EMOJI.get(a.get('urgence','?'),'?')} {m['subject']}",
            f"De     : {m['from']}",
            f"Date   : {m['date']}",
            f"Resume : {a.get('resume','')}",
            f"Action : {a.get('action_suggeree','')}",
            "\n--- BROUILLON ---",
            b["brouillon"],
            "\n" + "-"*65,
        ]
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Export TXT  : {filename}")

if BROUILLONS:
    export_html(BROUILLONS, "brouillons_agent.html")
    export_txt(BROUILLONS,  "brouillons_agent.txt")
    print("\nOuvrir brouillons_agent.html dans le navigateur")
else:
    print("Aucun brouillon - relancer Etape 8")

Export HTML : brouillons_agent.html
Export TXT  : brouillons_agent.txt

Ouvrir brouillons_agent.html dans le navigateur


---
## Etape 10 — Lancer l'agent complet (tout en un)

In [35]:
def run_agent_complet(gmail_user, gmail_password, n_mails=10, unread_only=False):
    print("="*60)
    print("AGENT MAIL - DEMARRAGE")
    print("="*60)

    ok, _ = check_ollama()
    if not ok:
        print("Ollama non disponible")
        return None, None

    r = GmailReader(gmail_user, gmail_password)
    if not r.connect(): return None, None
    mails = r.fetch(n=n_mails, unread_only=unread_only)
    r.disconnect()
    if not mails: print("Aucun mail"); return None, None

    print(f"\nTriage de {len(mails)} mails...")
    rank = {"critique":0,"haute":1,"normale":2,"faible":3}
    for i, mail in enumerate(mails):
        print(f"  [{i+1}/{len(mails)}] {mail['subject'][:50]}...", end=" ", flush=True)
        try:
            mail["analyse"] = trier_mail(mail)
            print(URGENCY_EMOJI.get(mail["analyse"].get("urgence","normale"),"🟡"))
        except Exception as e:
            print(f"erreur {e}")
            mail["analyse"] = {"urgence":"normale","categorie":"information",
                                "resume":mail["subject"][:120],
                                "action_suggeree":"Lire","delai_reponse":"cette_semaine"}

    for mail in [m for m in mails if m["attachments"]]:
        mail["pj_content"] = analyser_pj_mail(mail)

    prioritaires = sorted(
        [m for m in mails
         if m.get("analyse",{}).get("urgence") in ("critique","haute")
         or m.get("analyse",{}).get("categorie") == "action_requise"],
        key=lambda m: rank.get(m.get("analyse",{}).get("urgence","normale"),2)
    ) or mails[:3]

    print(f"\nBrouillons pour {len(prioritaires)} mails prioritaires...")
    brouillons = []
    for mail in prioritaires:
        try:
            draft = generer_reponse(mail)
            mail["brouillon"] = draft
            brouillons.append({"mail": mail, "brouillon": draft})
            print(f"  OK : {mail['subject'][:55]}")
        except Exception as e:
            print(f"  Erreur : {e}")

    if brouillons:
        export_html(brouillons, "brouillons_agent.html")
        export_txt(brouillons,  "brouillons_agent.txt")

    urgences = {}
    for m in mails:
        u = m.get("analyse",{}).get("urgence","normale")
        urgences[u] = urgences.get(u, 0) + 1

    print("\n" + "="*60)
    print("AGENT TERMINE")
    print("="*60)
    print(f"  Mails analyses : {len(mails)}")
    for u, c in sorted(urgences.items(), key=lambda x: rank.get(x[0],2)):
        print(f"  {URGENCY_EMOJI.get(u,'?')} {u:<12}: {c}")
    print(f"  Brouillons     : {len(brouillons)}")
    print(f"  Export         : brouillons_agent.html / .txt")
    return mails, brouillons

print("run_agent_complet() defini")
print()
print("Lancer l'agent :")
print("  mails, brouillons = run_agent_complet(GMAIL_USER, GMAIL_APP_PASSWORD)")

run_agent_complet() defini

Lancer l'agent :
  mails, brouillons = run_agent_complet(GMAIL_USER, GMAIL_APP_PASSWORD)


---
## Troubleshooting

| Erreur | Cause | Solution |
|---|---|---|
| `AUTHENTICATE failed` | Mauvais App Password | Regenerer sur myaccount.google.com |
| `IMAP disabled` | IMAP non active | Gmail Settings -> Enable IMAP |
| `Connection refused :11434` | Ollama non demarre | `ollama serve` |
| `model not found` | Modele absent | `ollama pull llama3.2:3b` |
| `JSONDecodeError` | JSON mal forme par le LLM | Essayer `qwen2.5:7b` (meilleur JSON) |
| `SSL CERTIFICATE_VERIFY_FAILED` | macOS sans certs | `pip install certifi` |

### Changer de modele (cellule Config)
```python
OLLAMA_MODEL = "llama3.1:8b"       # Meilleur equilibre
OLLAMA_MODEL = "mistral-nemo:12b"  # Meilleur en francais
OLLAMA_MODEL = "qwen2.5:7b"        # Meilleur pour le JSON structure
```

### Compte Google Workspace
L'administrateur doit activer IMAP et les App Passwords dans la console admin.